# Applied RL in 15 minutes

`decisionrl` is reinforcement learning for **operational decisions** — pricing, inventory, energy, queues, supply chains. This notebook trains agents on three of them and compares each to the classic operations-research baseline, so you can see where RL *matches* the textbook optimum and where it *beats* it.

Runs on CPU in a few minutes. Install with `pip install decisionrl` (add `[gym]` for Gymnasium envs).

In [ ]:
# !pip install decisionrl
from decisionrl import baselines as B
from decisionrl.algorithms import PPO, DQN
from decisionrl.envs import InventoryManagement, QueueAdmissionControl, NonstationaryInventory
from decisionrl.training import evaluate_policy
from decisionrl.utils import set_seed

## 1. Inventory management — RL *recovers* the optimal base-stock policy

Order stock under stochastic demand, trading off holding cost vs stockouts. The optimal policy is the newsvendor **base-stock** level — a provably optimal closed form. A PPO agent, given no domain knowledge, should *match* it.

In [ ]:
set_seed(0)
agent = PPO(InventoryManagement(), n_steps=1024, batch_size=64, n_epochs=10, seed=0)
agent.learn(40_000)
learned = evaluate_policy(agent, InventoryManagement(), n_episodes=30, seed=1)[0]

s_star = B.analytic_base_stock_level(InventoryManagement())
optimal = B.rollout_return(InventoryManagement, B.base_stock(s_star))
print(f'PPO:          {learned:.1f}')
print(f'base-stock S*={s_star} (optimal): {optimal:.1f}')
print('=> RL recovers the operations-research optimum from scratch.')

## 2. Queue admission control — RL *beats* the best fixed threshold

Admit or shed each arriving job to protect a finite buffer. The naive default is *admit everything*; a smarter classic baseline is the best fixed *value threshold*. RL learns a congestion-dependent policy that beats both.

In [ ]:
set_seed(0)
agent = PPO(QueueAdmissionControl(), n_steps=512, batch_size=64, n_epochs=10, seed=0)
agent.learn(30_000)
learned = evaluate_policy(agent, QueueAdmissionControl(), n_episodes=30, seed=1)[0]

admit_all = B.rollout_return(QueueAdmissionControl, B.admit_all())
_, best_threshold = B.best_value_threshold(QueueAdmissionControl)
print(f'PPO:                 {learned:.1f}')
print(f'best value threshold: {best_threshold:.1f}')
print(f'admit-all (naive):    {admit_all:.1f}')

## 3. Non-stationary inventory — where the *formula breaks*

This is the case that justifies RL. The demand rate switches between a low and a high regime, so **no single base-stock level is right**. A DQN agent reads recent demand and adapts its order-up-to level — beating the best fixed base-stock by a clear margin.

(Base-stock is provably optimal only for *stationary* demand. When that assumption breaks, reach for learning; when it holds — as in section 1 — use the formula.)

In [ ]:
set_seed(0)
agent = DQN(NonstationaryInventory(), learning_rate=5e-4, buffer_size=50_000,
            learning_starts=1000, target_update_interval=500, seed=0)
agent.learn(100_000)
learned = evaluate_policy(agent, NonstationaryInventory(), n_episodes=40, seed=1)[0]

_, best_fixed = B.best_base_stock(NonstationaryInventory)
print(f'DQN (adaptive):        {learned:.1f}')
print(f'best fixed base-stock: {best_fixed:.1f}')
print(f'=> RL wins by {100*(learned-best_fixed)/abs(best_fixed):.0f}% where the closed form cannot adapt.')

## Takeaways

- On **stationary, fully-observed** problems, RL matches the classic tool — use the solver/formula.
- On **non-stationary / partially-observed / coupled** problems, the closed form breaks and a learned policy pulls ahead.
- Every applied env ships with its baseline in `decisionrl.baselines`, so you can always check RL against the honest reference.

Under the hood `decisionrl` is a full RL library (DQN/PPO/SAC/TRPO, offline, model-based, multi-agent, meta-RL) — see the [docs](https://denisdrobyshev.github.io/decisionrl/).